## Data Loading

In [ ]:
import soccerdata as sd
print(sd.FBref.available_leagues())

In [ ]:
import soccerdata as sd
import pandas as pd

all_leagues = [
    
]

all_stats = []
for league in all_leagues:
    try:
        fbref = sd.FBref(leagues=[league], seasons=2025)
        s = fbref.read_player_season_stats(stat_type="standard")
        all_stats.append(s)
        print(f"{league}: {s.shape[0]} players")
    except Exception as e:
        print(f"{league}: ERROR — {e}")

stats = pd.concat(all_stats)
print(f"\nTotal players: {stats.shape[0]}")

# Coverage check
players = pd.read_csv("../data/processed/player_fixtures.csv")
wc_players = players[["player", "team", "position"]].drop_duplicates()

stats_names = set(stats.index.get_level_values("player").str.lower().str.strip())
wc_players["name_lower"] = wc_players["player"].str.lower().str.strip()

matched = wc_players[wc_players["name_lower"].isin(stats_names)]
unmatched = wc_players[~wc_players["name_lower"].isin(stats_names)]

print(f"Matched: {len(matched)} ({len(matched)/len(wc_players)*100:.1f}%)")
print(f"Unmatched: {len(unmatched)} ({len(unmatched)/len(wc_players)*100:.1f}%)")

match_rate_by_team = (
    wc_players.groupby("team")
    .apply(lambda g: g["name_lower"].isin(stats_names).mean())
    .sort_values()
)
print("\nTeams with lowest match rates:")
print(match_rate_by_team.head(20))
print("\nTeams with highest match rates:")
print(match_rate_by_team.tail(10))
